# 02 · Inheritance, Method Overriding, `super()` & Polymorphism

**Module 06 – Object-Oriented Programming** · Estimated time: 55 min

### 🎯 What you will learn
- Creating subclasses that inherit from a parent class
- Overriding methods, and extending them with `super()`
- `isinstance()` and `issubclass()`
- Multiple inheritance and the **MRO** (method resolution order)
- Polymorphism and **duck typing**
- Composition vs inheritance ("has-a" vs "is-a")

## 1. Inheritance basics

A **child class** (subclass) inherits attributes and methods from its **parent class** (superclass), then specialises or extends them.

In [1]:
class Animal:                        # parent / base / superclass
    def __init__(self, name):
        self.name = name
        print("Animal created")

    def who_am_i(self):
        print("I am an animal")

    def eat(self):
        print("Eating")

    def speak(self):
        print("Generic animal sound")


class Dog(Animal):                   # Dog inherits from Animal
    def __init__(self, name):
        super().__init__(name)       # run Animal's __init__ first
        print("Dog created")

    def who_am_i(self):              # override
        print("I am a dog")

    def bark(self):                  # new method
        print("Woof!")

d = Dog("Rover")
d.who_am_i()      # overridden
d.eat()           # inherited
d.bark()          # new
print(d.name)     # set by Animal.__init__

Animal created
Dog created
I am a dog
Eating
Woof!
Rover


## 2. Method overriding (📘)

When a child defines a method with the same name as the parent's, the **child's version wins** for child objects.

In [2]:
class Dog(Animal):
    def speak(self):
        print("Woof!")

class Cat(Animal):
    def speak(self):
        print("Meow!")

my_dog = Dog("Rover")
my_cat = Cat("Fluffy")
print(my_dog.name, my_cat.name)      # both inherited 'name'
my_dog.speak()
my_cat.speak()

Animal created
Animal created
Rover Fluffy
Woof!
Meow!


## 3. `super()`: extend, don't replace

`super()` gives access to the **parent's** methods. Most often used to call the parent's `__init__` and then add extra attributes:

In [3]:
class Animal:
    def __init__(self, name):
        self.name = name
    def describe(self):
        return f"{self.name} is an animal"

class Bird(Animal):
    def __init__(self, name, wingspan):
        super().__init__(name)          # let Animal set the name
        self.wingspan = wingspan        # add a Bird-specific attribute

    def describe(self):
        base = super().describe()       # reuse the parent's logic
        return f"{base} with a {self.wingspan} cm wingspan"

my_bird = Bird("Tweety", 10)
print(my_bird.name, my_bird.wingspan)
print(my_bird.describe())

Tweety 10
Tweety is an animal with a 10 cm wingspan


❌ **Forgetting `super().__init__()`** means the parent's attributes never get created:

In [4]:
class BrokenBird(Animal):
    def __init__(self, name, wingspan):
        self.wingspan = wingspan       # forgot super().__init__(name)

BrokenBird("Oops", 5).name             # AttributeError

AttributeError: 'BrokenBird' object has no attribute 'name'

> ⚠️ **Old (Python 2):** `super(Bird, self).__init__(name)`. In Python 3, just `super().__init__(name)`.

## 4. `isinstance()` and `issubclass()`

In [5]:
print(isinstance(my_bird, Bird), isinstance(my_bird, Animal), isinstance(my_bird, object))
print(issubclass(Bird, Animal), issubclass(Animal, Bird))
print(Bird.__bases__, Bird.__mro__)

True True True
True False
(<class '__main__.Animal'>,) (<class '__main__.Bird'>, <class '__main__.Animal'>, <class 'object'>)


Every class ultimately inherits from `object`.

## 5. Polymorphism: one interface, many forms

Different classes can share method names and be used **interchangeably**:

In [6]:
class Dog:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return self.name + " says Woof!"

class Cat:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return self.name + " says Meow!"

niko, felix = Dog("Niko"), Cat("Felix")
for pet in [niko, felix]:
    print(pet.speak())

def pet_speak(pet):
    print(pet.speak())

pet_speak(niko)
pet_speak(felix)

Niko says Woof!
Felix says Meow!
Niko says Woof!
Felix says Meow!


### Abstract-style base class (📗)
A common pattern: a base class that defines the interface and expects subclasses to implement it:

In [7]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        raise NotImplementedError("Subclass must implement abstract method")

class Dog(Animal):
    def speak(self):
        return self.name + " says Woof!"

class Cat(Animal):
    def speak(self):
        return self.name + " says Meow!"

print(Dog("Fido").speak(), Cat("Isis").speak())
try:
    Animal("generic").speak()
except NotImplementedError as e:
    print("NotImplementedError:", e)

Fido says Woof! Isis says Meow!
NotImplementedError: Subclass must implement abstract method


(Lesson 06 shows the proper tool for this: `abc.ABC` + `@abstractmethod`.)

### Duck typing
*"If it walks like a duck and quacks like a duck, it's a duck."* Python doesn't care about the class, only that the object has the right method:

In [8]:
class Robot:
    def speak(self):
        return "Beep boop"

for thing in [Dog("Rex"), Cat("Tom"), Robot()]:     # Robot isn't an Animal at all
    print(thing.speak())

print(len("abc"), len([1, 2]), len({"a": 1}))        # len() is polymorphic too

Rex says Woof!
Tom says Meow!
Beep boop
3 2 1


## 6. Multi-level and multiple inheritance

In [9]:
class Vehicle:
    def start(self):
        return "Starting vehicle"

class Car(Vehicle):
    def honk(self):
        return "Beep!"

class SportsCar(Car):                     # multi-level: SportsCar → Car → Vehicle
    def turbo(self):
        return "Turbo on!"

s = SportsCar()
print(s.start(), s.honk(), s.turbo())

Starting vehicle Beep! Turbo on!


In [10]:
class Flyer:
    def move(self):
        return "flying"
    def fly(self):
        return "I can fly"

class Swimmer:
    def move(self):
        return "swimming"
    def swim(self):
        return "I can swim"

class Duck(Flyer, Swimmer):               # multiple inheritance
    pass

d = Duck()
print(d.fly(), d.swim(), d.move())         # move() → Flyer's, listed first
print([cls.__name__ for cls in Duck.__mro__])

I can fly I can swim flying
['Duck', 'Flyer', 'Swimmer', 'object']


### The MRO (Method Resolution Order)
Python searches classes in the order shown by `Class.__mro__` (C3 linearisation): the class itself, then parents left-to-right, then their parents, ..., ending with `object`. `super()` follows the MRO, which lets cooperative multiple inheritance work.

> 💡 Use multiple inheritance sparingly; small **mixin** classes (e.g. `JsonMixin`, `LoggingMixin`) are the common, healthy use.

## 7. Composition vs inheritance

- **Inheritance = "is-a"**: a `Dog` *is an* `Animal`
- **Composition = "has-a"**: a `Car` *has an* `Engine`

Prefer composition when the relationship isn't truly "is-a"; it's more flexible.

In [11]:
class Engine:
    def __init__(self, hp):
        self.hp = hp
    def start(self):
        return f"{self.hp}hp engine roars"

class Car:
    def __init__(self, model, hp):
        self.model = model
        self.engine = Engine(hp)          # Car HAS an Engine
    def start(self):
        return f"{self.model}: {self.engine.start()}"

print(Car("Model S", 670).start())

Model S: 670hp engine roars


## 📝 Summary
- `class Child(Parent):` inherits everything; override by redefining.
- `super()` calls the parent's version, especially `super().__init__(...)`.
- `isinstance`/`issubclass`; every class derives from `object`.
- Polymorphism + duck typing: write code against behaviour, not concrete classes.
- Multiple inheritance follows the MRO; prefer composition for "has-a".

## 🏋️ Practice
1. Create `Shape` with `area()` raising `NotImplementedError`; subclasses `Rectangle`, `Circle`, `Triangle`. Loop over a list of shapes and print areas.
2. Create `Employee(name, salary)` and `Manager(Employee)` that adds `team` (a list) and overrides `__str__` using `super().__str__()`.
3. Print the MRO of `class D(B, C)` where `B(A)` and `C(A)`.

<details><summary>Show solutions</summary>

```python
import math
class Shape:
    def area(self): raise NotImplementedError
class Rectangle(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h
class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2
class Triangle(Shape):
    def __init__(self, b, h): self.b, self.h = b, h
    def area(self): return 0.5 * self.b * self.h
for s in [Rectangle(2, 3), Circle(1), Triangle(4, 5)]:
    print(type(s).__name__, round(s.area(), 2))

class Employee:
    def __init__(self, name, salary): self.name, self.salary = name, salary
    def __str__(self): return f"{self.name} (₹{self.salary})"
class Manager(Employee):
    def __init__(self, name, salary, team=None):
        super().__init__(name, salary)
        self.team = team or []
    def __str__(self): return super().__str__() + f", manages {len(self.team)}"

class A: pass
class B(A): pass
class C(A): pass
class D(B, C): pass
print([k.__name__ for k in D.__mro__])   # D B C A object
```
</details>